# Temporal CV leakage: impressive metric, wrong protocol

Synthetic daily demand series generated below with a fixed seed, so the notebook runs offline.
The same model and the same features are scored twice. Only the cross-validation protocol changes.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_validate

In [2]:
rng = np.random.default_rng(7)
n = 1460
t = np.arange(n)
demand = (200 + 0.15 * t + 25 * np.sin(2 * np.pi * t / 365.25)
          + 8 * np.sin(2 * np.pi * t / 7) + np.cumsum(rng.normal(0, 3, n)))
dates = pd.date_range("2021-01-01", periods=n, freq="D")
df = pd.DataFrame({"date": dates, "demand": demand})

In [3]:
df["lag_1"] = df["demand"].shift(1)
df["lag_7"] = df["demand"].shift(7)
df["rolling_7"] = df["demand"].shift(1).rolling(7).mean()
df["day_of_week"] = df["date"].dt.dayofweek
df = df.dropna().reset_index(drop=True)
features = ["lag_1", "lag_7", "rolling_7", "day_of_week"]
X = df[features]
y = df["demand"]
model = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)
scoring = ("r2", "neg_mean_absolute_error")

## Before: shuffled 5-fold CV

Each fold trains on days that come after the days it is tested on.
The lag features make neighbouring days near-duplicates, so the model interpolates instead of forecasting.

In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=0)
shuffled = cross_validate(model, X, y, cv=kf, scoring=scoring)
print(f"shuffled KFold: R2={shuffled['test_r2'].mean():.3f} MAE={-shuffled['test_neg_mean_absolute_error'].mean():.2f}")

shuffled KFold: R2=0.992 MAE=2.98


## After: TimeSeriesSplit

Every fold trains only on the past and tests on the following block of days.

In [5]:
tscv = TimeSeriesSplit(n_splits=5)
temporal = cross_validate(model, X, y, cv=tscv, scoring=scoring)
print(f"TimeSeriesSplit: R2={temporal['test_r2'].mean():.3f} MAE={-temporal['test_neg_mean_absolute_error'].mean():.2f}")

TimeSeriesSplit: R2=0.465 MAE=9.82


## What changed

The data, features and model are identical. The difference between the two printed scores is the leakage.
`waldean check` flags the shuffled protocol statically, without running this notebook. The metric gap above is computed here, not claimed by Waldean.